# Spike phase coupling

This notebook calculates the AIS-AP and RA-AP firing rate during the cycles of oscillations.

## Initialization - don't change

In [ ]:
import sys
sys.path.append('..')

from workflows.run_util.io import collect_paths_from_table
from param_classes.workflows.spike_phase_coupling import (
    SpikePhaseCouplingWorkflowParams
)
from workflows.spike_phase_coupling import run_analysis, run_grouped_analysis

params = SpikePhaseCouplingWorkflowParams()

## Color settings

Set the colors that will be used for coloring the markers on the strip plot and the cellwise average firing frequency curve.

In [ ]:
params.palette = {
    'cell_type': {
        'PYR_L2-3': '#803300ff',
        'PYR_L4-6': '#502d16ff',
        'RSI_L2-3': '#ff48a3ff',
        'RSI_L4-6': '#800080ff',
        'FSI_L2-3': '#d54500ff',
        'FSI_L4-6': '#7a0000ff'
    },
    'spike_type': {
        'RA-AP': '#00ff00'
    }
}

## Data settings

Set raw input dir. You need to give the base path where the group directories can be found.

In [ ]:
params.data.raw_data.path='E:\\Data\\BogiSejtek\\20251007\\data\\invivo'
params.data.raw_data.extension='mat'

Set the base path of the directory where csv files containing the spike features and categories can be found.
> It needs to be the same structure as the `raw_data` directory.

In [ ]:
params.data.spike_features.path = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\SpikeFeatures'
params.data.spike_features.extension = 'csv'

## Input-output settings

Set the Excel table containing the cell attributes, grouping and selector information.

In [ ]:
params.input = {
    'mode': 'table',
    'table_path': 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\data_invivo.xlsx',
    'select_rules': [
        {'selected': 1}
    ]
}

Set the output base directory path. All results of this analysis (cellwise aggregated tables, grouped results, figures, statistics) is going to saved there.

In [ ]:
params.output.path = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\SpikePhaseCoupling'

## Analysis parameters

Here you can set the histogram parameters.

 - `spike_types`: the list of the spike types to to analyze during the cycles

In [ ]:
params.analysis.spike_types = ['AIS-AP', 'RA-AP']

 - `episode_name`: the name of the oscillation episode
 - `marker_name`: the name of the marker that marks the cycle ends

In [ ]:
params.analysis.episode_name = 'spdl'
params.analysis.marker_name = 'spdl_0'

 - `values`: unit of the histogram, possible values: `'count'`, `'freq'`, `'prob'`
 - `num_bins`: number of the bins to use for each cycle

In [ ]:
params.analysis.values = 'freq'
params.analysis.num_bins = 90

- `smooth_size`: the sigma parameter of the Gaussian smoothing kernel; set to 0 if you want raw histogram.

In [ ]:
params.analysis.smooth_size = 1

## Plot parameters

 - `xlim`: limit of the x axis, interpreted as circular axis, i.e. if the x limit extends below 0 or above 360, the histogram values will be repeated and shifted.

In [ ]:
params.analysis.xlim = (-360, 360)

 - `ylim`: limit of the y axis

In [ ]:
params.analysis.ylim = (0, 100)

You can customize the raster plot.
 - `show_raster`: set `True` if you want to show the raster above the histogram
 - `raster_size`: length of the raster tick
 - `raster_band_color`: color of the band to mark interval with any Ra-Ap spikes

In [ ]:
params.analysis.show_raster = True
params.analysis.raster_size = 0.04
params.analysis.raster_band_color = '#bbbbbb'

### Cellwise average parameters

 - `interval_condition`: if you leave empty (`[]`), all interals will be used. If you put values in the list, only those intervals will be used to calculate the average where that spike type was present

In [ ]:
params.analysis.interval_condition = ['RA-AP']

## Run analysis

Collect input files according to the table.

In [ ]:
input_files = collect_paths_from_table(params.data, params.input)
print(f"{len(input_files)} files selected for analysis.")

### Individual analysis

Perform individual analysis and save the results.

In [ ]:
cell_results = run_analysis(input_files, params)

Sort cells into groups by the `group` column of the Excel table.

In [ ]:
group_names = list(set([file.info.cell_type for file in input_files]))
groups = []
for group_name in group_names:
    group_files = [
        file for file in cell_results
        if file.info.cell_type == group_name
    ]
    groups.append((group_name, group_files))
print(f"{len(groups)} groups found for aggregation:")
for group in groups:
    print(f"\t- {group[0]}")

Aggregate each cells within groups. It will
 - average triggered histograms that satisfies the condition
 - detect maximum time and value
 - plot and save the aggregation result

In [ ]:
# group_aggregates = []
# for group_name, group_files in groups:
#     group_aggregates.append(run_grouped_analysis(group_files, group_name, params))

### Grouped analysis

In [ ]:
# from workflows.histogram_grouping import run_grouped_analysis
# params.grouping.flag = 1
# params.grouping.path = 'C:\\tmp\\Bozso2025\\PutRAAP\\Results\\New\\data_invivo.xlsx'
# params.grouping.column_name = 'group'
# params.grouping.groups =  ['sw']
# params.grouping.spike_types = ['AIS-AP', 'RA-AP']
# params.grouping.conditions = ['RA-AP']

# run_grouped_analysis(
#     config=params.grouping,
#     cell_results=cell_results
# )